<a href="https://colab.research.google.com/github/Sayak-coder/spam_mssg_detector_using_LSTM/blob/main/spam_mssg_classification_using_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [10]:
df=pd.read_csv('/content/spam.csv')

In [11]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [48]:
df['Category'].unique()

array(['ham', 'spam'], dtype=object)

In [54]:
df['Category']=df['Category'].map({'spam':1,'ham':0})

In [12]:
df.isnull().sum()

,0
Category,0
Message,0


In [14]:
df.shape

(5572, 2)

In [15]:
import tensorflow as tf
tf.__version__

'2.20.0'

In [17]:
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [55]:
X=df.drop('Category',axis=1)
y=df['Category']

In [19]:
voc_size=5000

In [20]:
import nltk
import re
from nltk.corpus import stopwords

In [21]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [25]:
X

,Message
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."
...,...
5567,This is the 2nd time we have tried 2 contact u...
5568,Will ü b going to esplanade fr home?
5569,"Pity, * was in mood for that. So...any other s..."
5570,The guy did some bitching but I acted like i'd...


In [26]:
len(X)

5572

In [27]:
from nltk.stem.porter import PorterStemmer
stemmer=PorterStemmer()
corpus=[]
for i in range(0,len(X)):
  review=re.sub('[^a-zA-Z]',' ',X['Message'][i])
  review=review.lower()
  review=review.split()

  review=(stemmer.stem(word) for word in review if not word in stopwords.words('english'))
  review=' '.join(review)
  corpus.append(review)

In [28]:
corpus

['go jurong point crazi avail bugi n great world la e buffet cine got amor wat',
 'ok lar joke wif u oni',
 'free entri wkli comp win fa cup final tkt st may text fa receiv entri question std txt rate c appli',
 'u dun say earli hor u c alreadi say',
 'nah think goe usf live around though',
 'freemsg hey darl week word back like fun still tb ok xxx std chg send rcv',
 'even brother like speak treat like aid patent',
 'per request mell mell oru minnaminungint nurungu vettam set callertun caller press copi friend callertun',
 'winner valu network custom select receivea prize reward claim call claim code kl valid hour',
 'mobil month u r entitl updat latest colour mobil camera free call mobil updat co free',
 'gonna home soon want talk stuff anymor tonight k cri enough today',
 'six chanc win cash pound txt csh send cost p day day tsandc appli repli hl info',
 'urgent week free membership prize jackpot txt word claim c www dbuk net lccltd pobox ldnw rw',
 'search right word thank breather

In [29]:
one_hot_rep=[one_hot(word,voc_size)for word in corpus]
one_hot_rep

[[1716,
  2776,
  4877,
  2904,
  3628,
  4007,
  826,
  228,
  3492,
  3816,
  2716,
  2466,
  2151,
  2437,
  2983,
  1427],
 [4304, 2350, 2022, 4213, 4418, 2185],
 [3005,
  2710,
  1163,
  3756,
  3703,
  4486,
  4685,
  3148,
  4264,
  2350,
  4478,
  2162,
  4486,
  4681,
  2710,
  4596,
  4813,
  1336,
  2453,
  3176,
  2278],
 [4418, 3678, 4876, 33, 2479, 4418, 3176, 957, 4876],
 [66, 2776, 4952, 815, 3170, 715, 378],
 [2326,
  1262,
  4569,
  4904,
  3149,
  4630,
  1819,
  194,
  1151,
  28,
  4304,
  645,
  4813,
  1599,
  382,
  2891],
 [522, 1749, 1819, 4935, 2636, 1819, 3503, 4774],
 [2956,
  4317,
  667,
  667,
  3973,
  457,
  3900,
  4583,
  4522,
  457,
  2688,
  4539,
  534,
  2060,
  457],
 [4225,
  796,
  2333,
  1826,
  4621,
  3647,
  285,
  1697,
  2008,
  3128,
  2008,
  47,
  1479,
  202,
  1542],
 [2119,
  2086,
  4418,
  543,
  964,
  438,
  4164,
  4578,
  2119,
  427,
  3005,
  3128,
  2119,
  438,
  3290,
  3005],
 [3081, 2764, 3459, 1879, 1674, 4018, 3694

In [30]:
print(corpus[0])
print(one_hot_rep[0])

go jurong point crazi avail bugi n great world la e buffet cine got amor wat
[1716, 2776, 4877, 2904, 3628, 4007, 826, 228, 3492, 3816, 2716, 2466, 2151, 2437, 2983, 1427]


In [31]:
max_length=30
embedded_doc=pad_sequences(one_hot_rep,padding='post',maxlen=max_length)
embedded_doc

array([[1716, 2776, 4877, ...,    0,    0,    0],
       [4304, 2350, 2022, ...,    0,    0,    0],
       [3005, 2710, 1163, ...,    0,    0,    0],
       ...,
       [2685, 3128, 4941, ...,    0,    0,    0],
       [2088, 1799, 2439, ...,    0,    0,    0],
       [4517, 4246, 1966, ...,    0,    0,    0]], dtype=int32)

In [37]:
embedded_doc[5001]

array([1361, 1819, 3351, 3128, 1611,  140, 4686,  881,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0], dtype=int32)

In [60]:
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=max_length))
model.add(LSTM(150))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [56]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(embedded_doc,y,test_size=0.2,random_state=45)

In [57]:
X_train.shape,X_test.shape

((4457, 30), (1115, 30))

In [58]:
y_train

,Category
3366,0
3022,0
1160,0
3778,1
585,0
...,...
4473,1
580,0
163,0
4703,0


In [61]:
epochs=10
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=epochs,batch_size=64)

Epoch 1/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 7s 67ms/step - accuracy: 0.9100 - loss: 0.2830 - val_accuracy: 0.9677 - val_loss: 0.1198
Epoch 2/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 6s 84ms/step - accuracy: 0.9854 - loss: 0.0594 - val_accuracy: 0.9803 - val_loss: 0.0706
Epoch 3/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 5s 64ms/step - accuracy: 0.9915 - loss: 0.0364 - val_accuracy: 0.9785 - val_loss: 0.0761
Epoch 4/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 5s 63ms/step - accuracy: 0.9953 - loss: 0.0219 - val_accuracy: 0.9821 - val_loss: 0.0853
Epoch 5/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 6s 76ms/step - accuracy: 0.9964 - loss: 0.0178 - val_accuracy: 0.9821 - val_loss: 0.0914
Epoch 6/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 5s 67ms/step - accuracy: 0.9962 - loss: 0.0190 - val_accuracy: 0.9812 - val_loss: 0.1129
Epoch 7/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 6s 84ms/step - accuracy: 0.9982 - loss: 0.0104 - val_accuracy: 0.9758 - val_loss: 0.1201
Epoch 8/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 9s 64ms/step - accuracy: 0.9991 - loss: 0.0063 - val_accuracy: 0.9776 - v

In [63]:
from tensorflow.keras.layers import Dropout
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=max_length))
model.add(Dropout(0.3))
model.add(LSTM(150))
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [66]:
epochs=2
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=epochs,batch_size=64)

Epoch 1/2
70/70 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step - accuracy: 0.9935 - loss: 0.0363 - val_accuracy: 0.9776 - val_loss: 0.1290
Epoch 2/2
70/70 ━━━━━━━━━━━━━━━━━━━━ 6s 86ms/step - accuracy: 0.9960 - loss: 0.0268 - val_accuracy: 0.9803 - val_loss: 0.1045


In [67]:
y_pred=model.predict(X_test)

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step


In [69]:
import numpy as np
y_pred=np.where(y_pred>0.6,1,0)

In [70]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_pred,y_test))

0.9802690582959641


In [71]:
from sklearn.metrics import classification_report
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99       984
           1       0.87      0.98      0.92       131

    accuracy                           0.98      1115
   macro avg       0.93      0.98      0.96      1115
weighted avg       0.98      0.98      0.98      1115



## README: SMS Spam Detection using LSTM

This project implements an SMS spam detection system using a Long Short-Term Memory (LSTM) neural network. The goal is to classify SMS messages as either 'ham' (legitimate) or 'spam' based on their textual content.

### 1. Project Overview

The notebook demonstrates the complete workflow for building a text classification model:
-   **Data Loading and Initial Exploration**: Loading the dataset and examining its structure.
-   **Data Preprocessing**: Cleaning text data, handling stopwords, stemming, and converting text into numerical representations suitable for a neural network.
-   **Model Building**: Constructing an LSTM-based deep learning model using TensorFlow/Keras.
-   **Training and Evaluation**: Training the model on preprocessed data and evaluating its performance using standard metrics.

### 2. Dataset

The dataset used is `spam.csv`, which contains SMS messages labeled as 'ham' or 'spam'.

-   **Source**: `/content/spam.csv`
-   **Columns**: `Category` (label) and `Message` (SMS text).
-   **Shape**: (5572, 2)

### 3. Dependencies

Key libraries used in this project include:
-   `pandas` for data manipulation.
-   `nltk` for natural language processing tasks (stopwords, stemming).
-   `tensorflow.keras` for building and training the LSTM model.
-   `sklearn` for data splitting and model evaluation metrics.

### 4. Data Preprocessing Steps

1.  **Load Data**: The `spam.csv` file is loaded into a pandas DataFrame.
2.  **Target Encoding**: The `Category` column is converted from categorical strings ('ham', 'spam') to numerical labels (0 for 'ham', 1 for 'spam').
3.  **Text Cleaning (`corpus` creation)**:
    -   Messages are converted to lowercase.
    -   Non-alphabetic characters are removed.
    -   Messages are tokenized into words.
    -   Stop words (common words like 'the', 'is', 'a') are removed using `nltk.corpus.stopwords`.
    -   Words are stemmed using `PorterStemmer` to reduce them to their root form (e.g., 'running' -> 'run').
4.  **One-Hot Encoding**: Each word in the cleaned `corpus` is converted into a numerical representation using `one_hot` encoding, based on a vocabulary size (`voc_size = 5000`).
5.  **Padding Sequences**: The one-hot encoded sequences are padded to a fixed maximum length (`max_length = 30`) to ensure uniform input size for the neural network. Padding is applied post-sequence.

### 5. Model Architecture (LSTM)

The model is a sequential Keras model with the following layers:

-   **Embedding Layer**: Maps integer-encoded words to dense vectors of fixed size (`embedding_vector_features = 40`).
    -   `model.add(Embedding(voc_size, embedding_vector_features, input_length=max_length))`
-   **Dropout Layer (0.3)**: Regularization layer to prevent overfitting.
-   **LSTM Layer**: A Long Short-Term Memory layer with 150 units, capable of learning long-term dependencies in sequences.
    -   `model.add(LSTM(150))`
-   **Dropout Layer (0.3)**: Another regularization layer.
-   **Dense Output Layer**: A single neuron with a `sigmoid` activation function for binary classification.
    -   `model.add(Dense(1, activation='sigmoid'))`

**Compilation**: The model is compiled with:
-   **Loss Function**: `binary_crossentropy` (suitable for binary classification).
-   **Optimizer**: `adam`.
-   **Metrics**: `accuracy`.

### 6. Training

The dataset is split into training and testing sets (80% training, 20% testing) using `train_test_split` with `random_state=45`.

The model is trained for a specified number of `epochs` (initially 10, then 2 in a refined model) with a `batch_size` of 64.

### 7. Evaluation

After training, the model's performance is evaluated on the test set:
-   **Predictions**: `y_pred = model.predict(X_test)` generates raw probability scores.
-   **Thresholding**: Predictions are converted to binary classes (0 or 1) using a threshold of 0.6 (`np.where(y_pred > 0.6, 1, 0)`).
-   **Accuracy Score**: `accuracy_score(y_pred, y_test)` calculates the overall accuracy.
-   **Classification Report**: `classification_report(y_pred, y_test)` provides detailed metrics like precision, recall, and f1-score for each class.

### 8. Results

The model achieved an accuracy of approximately **98.03%** on the test set. The classification report indicates strong performance, especially for the 'ham' class (0). The 'spam' class (1) also shows good recall, although precision is slightly lower, which is common in imbalanced datasets like spam detection where false positives are more tolerated than false negatives for spam.

**Classification Report:**
```
              precision    recall  f1-score   support

           0       1.00      0.98      0.99       984
           1       0.87      0.98      0.92       131

    accuracy                           0.98      1115
   macro avg       0.93      0.98      0.96      1115
weighted avg       0.98      0.98      0.98      1115
```

### 9. Usage

To run this project, simply execute the cells in the notebook sequentially. Ensure all necessary libraries are installed (`pandas`, `nltk`, `tensorflow`, `sklearn`). The `nltk` stopwords will be downloaded automatically if not already present.